In [1]:
import os
import numpy as np
import pandas as pd

DATA_DIR_CANDIDATES = [
    "/kaggle/input/siim-isic-melanoma-classification",
    "/kaggle/data/siim-isic-melanoma-classification",
    "/kaggle/input",
    "/kaggle/data",
]


def find_existing_path(rel_path: str):
    for base in DATA_DIR_CANDIDATES:
        p = os.path.join(base, rel_path)
        if os.path.exists(p):
            return p
    return None


train_path = find_existing_path("train.csv")
test_path = find_existing_path("test.csv")
sample_path = find_existing_path("sample_submission.csv")

if test_path is None or sample_path is None:
    raise FileNotFoundError(
        "Could not locate test.csv and/or sample_submission.csv in known Kaggle directories."
    )

train_df = pd.read_csv(train_path) if train_path is not None else None
test_df = pd.read_csv(test_path)
sample_sub = pd.read_csv(sample_path)

assert "image_name" in test_df.columns, "test.csv must contain image_name"
assert list(sample_sub.columns) == [
    "image_name",
    "target",
], "Unexpected sample_submission.csv columns"



In [2]:
if train_df is not None and "target" in train_df.columns:
    df_tr = train_df.copy()

    def _norm_cat(s: pd.Series) -> pd.Series:
        s = s.astype("string")
        s = s.fillna("unknown")
        s = s.str.strip()
        s = s.mask(s == "", "unknown")
        return s

    cat_cols_base = ["sex", "anatom_site_general_challenge", "patient_id"]
    for c in cat_cols_base:
        if c in df_tr.columns:
            df_tr[c] = _norm_cat(df_tr[c])

    if "diagnosis" in df_tr.columns:
        df_tr["diagnosis"] = _norm_cat(df_tr["diagnosis"])

    if "sex" in df_tr.columns and "anatom_site_general_challenge" in df_tr.columns:
        df_tr["sex_site"] = df_tr["sex"] + "||" + df_tr["anatom_site_general_challenge"]
    else:
        df_tr["sex_site"] = "unknown||unknown"

    if (
        "patient_id" in df_tr.columns
        and "anatom_site_general_challenge" in df_tr.columns
    ):
        df_tr["pid_site"] = (
            df_tr["patient_id"] + "||" + df_tr["anatom_site_general_challenge"]
        )
    else:
        df_tr["pid_site"] = "unknown||unknown"

    prior = float(np.clip(float(df_tr["target"].mean()), 1e-6, 1 - 1e-6))

    def logit(p):
        p = np.clip(p, 1e-6, 1 - 1e-6)
        return np.log(p / (1 - p))

    def sigmoid(x):
        return 1.0 / (1.0 + np.exp(-x))

    def make_oof_smoothed_rate(
        series: pd.Series,
        y: pd.Series,
        prior: float,
        m: float,
        n_splits: int = 5,
        seed: int = 42,
        group: pd.Series = None,
    ):
        series = series.astype("string").fillna("unknown")
        n = len(series)
        oof = np.empty(n, dtype="float64")

        rng = np.random.RandomState(seed)
        if group is None:
            fold_id = rng.randint(0, n_splits, size=n).astype(np.int16, copy=False)
        else:
            group = group.astype("string").fillna("unknown")
            g_codes, g_uniques = pd.factorize(group, sort=False)
            perm = rng.permutation(len(g_uniques))
            fold_per_guniq = (perm % n_splits).astype(np.int16, copy=False)
            fold_id = fold_per_guniq[g_codes]

        codes, uniques = pd.factorize(series, sort=False)
        codes = codes.astype(np.int32, copy=False)
        n_cat = len(uniques)

        y_arr = y.to_numpy(dtype=np.float64, copy=False)

        for f in range(n_splits):
            tr_mask = fold_id != f
            te_mask = ~tr_mask

            tr_codes = codes[tr_mask]
            cnt = np.bincount(tr_codes, minlength=n_cat).astype(np.float64, copy=False)
            pos = np.bincount(tr_codes, weights=y_arr[tr_mask], minlength=n_cat).astype(
                np.float64, copy=False
            )
            rate = (pos + m * prior) / (cnt + m)

            oof[te_mask] = rate[codes[te_mask]]

        return pd.Series(oof, index=series.index)

    def smoothed_map(series: pd.Series, y: pd.Series, prior: float, m: float):
        series = series.astype("string").fillna("unknown")
        stats = (
            y.groupby(series)
            .agg(["sum", "count"])
            .rename(columns={"sum": "pos", "count": "cnt"})
        )
        stats["rate"] = (stats["pos"] + m * prior) / (stats["cnt"] + m)
        return stats["rate"].to_dict()

    age_tr_num = pd.to_numeric(
        df_tr.get("age_approx", pd.Series([np.nan] * len(df_tr))), errors="coerce"
    )
    age_mean = (
        float(np.nanmean(age_tr_num)) if np.isfinite(np.nanmean(age_tr_num)) else 50.0
    )
    age_std = (
        float(np.nanstd(age_tr_num))
        if np.isfinite(np.nanstd(age_tr_num)) and float(np.nanstd(age_tr_num)) > 0
        else 10.0
    )

    age_fill = age_tr_num.fillna(age_mean)
    try:
        age_bin_tr = pd.qcut(age_fill, q=8, duplicates="drop")
        age_bin_edges = age_bin_tr.cat.categories
        age_bin_labels = list(range(len(age_bin_edges)))
        age_bin_tr = age_bin_tr.cat.rename_categories(age_bin_labels).astype(int)
        has_age_bins = True
    except Exception:
        age_bin_tr = pd.Series([0] * len(df_tr), index=df_tr.index)
        age_bin_edges = None
        has_age_bins = False

    if has_age_bins and "anatom_site_general_challenge" in df_tr.columns:
        df_tr["site_agebin"] = (
            df_tr["anatom_site_general_challenge"] + "||" + age_bin_tr.astype("string")
        )
    else:
        df_tr["site_agebin"] = "unknown||0"

    group_pid = df_tr["patient_id"] if "patient_id" in df_tr.columns else None

    oof_sex = (
        make_oof_smoothed_rate(
            df_tr["sex"], df_tr["target"], prior, m=200.0, group=group_pid
        )
        if "sex" in df_tr.columns
        else pd.Series([prior] * len(df_tr), index=df_tr.index, dtype="float64")
    )
    oof_site = (
        make_oof_smoothed_rate(
            df_tr["anatom_site_general_challenge"],
            df_tr["target"],
            prior,
            m=50.0,
            group=group_pid,
        )
        if "anatom_site_general_challenge" in df_tr.columns
        else pd.Series([prior] * len(df_tr), index=df_tr.index, dtype="float64")
    )
    oof_sexsite = make_oof_smoothed_rate(
        df_tr["sex_site"], df_tr["target"], prior, m=400.0, group=group_pid
    )

    oof_pid = (
        make_oof_smoothed_rate(
            df_tr["patient_id"], df_tr["target"], prior, m=800.0, group=None
        )
        if "patient_id" in df_tr.columns
        else pd.Series([prior] * len(df_tr), index=df_tr.index, dtype="float64")
    )

    oof_diag = (
        make_oof_smoothed_rate(
            df_tr["diagnosis"], df_tr["target"], prior, m=300.0, group=group_pid
        )
        if "diagnosis" in df_tr.columns
        else pd.Series([prior] * len(df_tr), index=df_tr.index, dtype="float64")
    )
    oof_agebin = (
        make_oof_smoothed_rate(
            age_bin_tr.astype("string"),
            df_tr["target"],
            prior,
            m=100.0,
            group=group_pid,
        )
        if has_age_bins
        else pd.Series([prior] * len(df_tr), index=df_tr.index, dtype="float64")
    )

    oof_siteage = (
        make_oof_smoothed_rate(
            df_tr["site_agebin"], df_tr["target"], prior, m=120.0, group=group_pid
        )
        if (has_age_bins and "anatom_site_general_challenge" in df_tr.columns)
        else pd.Series([prior] * len(df_tr), index=df_tr.index, dtype="float64")
    )

    # CHANGE (score improvement): compute OOF for pid_site with row-based folds (group=None).
    # Patient-grouped folds makes pid_site almost always unseen in-train for the held-out fold,
    # collapsing rates toward the prior and removing a potentially strong interaction signal.
    oof_pidsite = (
        make_oof_smoothed_rate(
            df_tr["pid_site"], df_tr["target"], prior, m=500.0, group=None
        )
        if ("pid_site" in df_tr.columns)
        else pd.Series([prior] * len(df_tr), index=df_tr.index, dtype="float64")
    )

    sex_map_full = (
        smoothed_map(df_tr["sex"], df_tr["target"], prior, m=200.0)
        if "sex" in df_tr.columns
        else {}
    )
    site_map_full = (
        smoothed_map(
            df_tr["anatom_site_general_challenge"], df_tr["target"], prior, m=50.0
        )
        if "anatom_site_general_challenge" in df_tr.columns
        else {}
    )
    sexsite_map_full = smoothed_map(df_tr["sex_site"], df_tr["target"], prior, m=400.0)
    pid_map_full = (
        smoothed_map(df_tr["patient_id"], df_tr["target"], prior, m=800.0)
        if "patient_id" in df_tr.columns
        else {}
    )
    diag_map_full = (
        smoothed_map(df_tr["diagnosis"], df_tr["target"], prior, m=300.0)
        if "diagnosis" in df_tr.columns
        else {}
    )

    siteage_map_full = (
        smoothed_map(df_tr["site_agebin"], df_tr["target"], prior, m=120.0)
        if (has_age_bins and "anatom_site_general_challenge" in df_tr.columns)
        else {}
    )

    pidsite_map_full = (
        smoothed_map(df_tr["pid_site"], df_tr["target"], prior, m=500.0)
        if ("pid_site" in df_tr.columns)
        else {}
    )

    if has_age_bins:
        agebin_map_full = smoothed_map(
            age_bin_tr.astype("string"), df_tr["target"], prior, m=100.0
        )
        low_bin = int(age_bin_tr.min())
        high_bin = int(age_bin_tr.max())
        p_low = float(np.clip(agebin_map_full.get(str(low_bin), prior), 1e-6, 1 - 1e-6))
        p_high = float(
            np.clip(agebin_map_full.get(str(high_bin), prior), 1e-6, 1 - 1e-6)
        )
        age_slope = float(
            np.clip(
                (logit(p_high) - logit(p_low)) / max(1.0, (high_bin - low_bin)),
                -0.35,
                0.35,
            )
        )
    else:
        agebin_map_full = {}
        age_slope = 0.0

    base = logit(prior)
    age_z_tr = (age_fill - age_mean) / age_std

    y_tr = df_tr["target"].values.astype(np.int8)

    d_sex = logit(oof_sex.values) - base
    d_site = logit(oof_site.values) - base
    d_sexsite = logit(oof_sexsite.values) - base
    d_agebin = logit(oof_agebin.values) - base
    d_pid = logit(oof_pid.values) - base
    d_diag = logit(oof_diag.values) - base
    d_agez = age_slope * age_z_tr.values
    d_siteage = logit(oof_siteage.values) - base
    d_pidsite = logit(oof_pidsite.values) - base

    def auc_from_scores_tieavg(y_true: np.ndarray, score: np.ndarray) -> float:
        y_true = np.asarray(y_true, dtype=np.int8)
        score = np.asarray(score, dtype=np.float64)

        n_pos = int(y_true.sum())
        n = y_true.size
        n_neg = n - n_pos
        if n_pos == 0 or n_neg == 0:
            return 0.5

        order = np.argsort(score, kind="mergesort")
        s_sorted = score[order]

        diff = np.empty(n, dtype=bool)
        diff[0] = True
        diff[1:] = s_sorted[1:] != s_sorted[:-1]
        starts = np.flatnonzero(diff)
        ends = np.r_[starts[1:], n]

        lens = (ends - starts).astype(np.int64, copy=False)
        avg_ranks = (starts + 1 + ends) / 2.0
        ranks_sorted = np.repeat(avg_ranks, lens).astype(np.float64, copy=False)

        ranks = np.empty(n, dtype=np.float64)
        ranks[order] = ranks_sorted

        sum_ranks_pos = float(ranks[y_true == 1].sum())
        auc = (sum_ranks_pos - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg)
        return float(auc)

    w_sex_grid = np.array([0.45, 0.60, 0.75], dtype=np.float64)
    w_site_grid = np.array([0.45, 0.60, 0.75], dtype=np.float64)
    w_sexsite_grid = np.array([0.35, 0.55, 0.75], dtype=np.float64)
    w_agebin_grid = np.array([0.65, 0.90, 1.15], dtype=np.float64)
    w_pid_grid = np.array([0.25, 0.35, 0.50], dtype=np.float64)
    w_diag_grid = np.array([0.00, 0.05, 0.10], dtype=np.float64)
    w_agez_grid = np.array([0.15, 0.25, 0.35], dtype=np.float64)
    w_siteage_grid = np.array([0.15, 0.30, 0.45], dtype=np.float64)
    w_pidsite_grid = np.array([0.00, 0.15, 0.30], dtype=np.float64)

    # CHANGE (score improvement): actually search over scale_grid too, and store best_scale.
    # This keeps the same model family and inference (sigmoid(scale*score)), but aligns
    # inference with what was selected on OOF instead of hard-coding 1.0.
    scale_grid = np.array([0.6, 0.8, 1.0, 1.2], dtype=np.float64)

    best_auc = -1.0
    best_params = (0.60, 0.60, 0.55, 0.90, 0.35, 0.05, 0.25, 0.30, 0.15, 1.0)

    for w_sex in w_sex_grid:
        for w_site in w_site_grid:
            for w_sexsite in w_sexsite_grid:
                for w_agebin in w_agebin_grid:
                    for w_pid in w_pid_grid:
                        for w_diag in w_diag_grid:
                            for w_agez in w_agez_grid:
                                for w_siteage in w_siteage_grid:
                                    for w_pidsite in w_pidsite_grid:
                                        score_oof = (
                                            base
                                            + w_sex * d_sex
                                            + w_site * d_site
                                            + w_sexsite * d_sexsite
                                            + w_agebin * d_agebin
                                            + w_pid * d_pid
                                            + w_diag * d_diag
                                            + w_agez * d_agez
                                            + w_siteage * d_siteage
                                            + w_pidsite * d_pidsite
                                        )

                                        for sc in scale_grid:
                                            auc_s = auc_from_scores_tieavg(
                                                y_tr, sc * score_oof
                                            )
                                            if auc_s > best_auc:
                                                best_auc = auc_s
                                                best_params = (
                                                    float(w_sex),
                                                    float(w_site),
                                                    float(w_sexsite),
                                                    float(w_agebin),
                                                    float(w_pid),
                                                    float(w_diag),
                                                    float(w_agez),
                                                    float(w_siteage),
                                                    float(w_pidsite),
                                                    float(sc),
                                                )

    (
        w_sex,
        w_site,
        w_sexsite,
        w_agebin,
        w_pid,
        w_diag,
        w_agez,
        w_siteage,
        w_pidsite,
        best_scale,
    ) = best_params

else:
    prior = 0.5
    age_mean, age_std, age_slope = 50.0, 10.0, 0.0
    age_bin_edges = None
    has_age_bins = False
    sex_map_full = {}
    site_map_full = {}
    sexsite_map_full = {}
    pid_map_full = {}
    diag_map_full = {}
    agebin_map_full = {}
    siteage_map_full = {}
    pidsite_map_full = {}
    w_sex, w_site, w_sexsite, w_agebin, w_pid, w_diag, w_agez, w_siteage, w_pidsite = (
        0.60,
        0.60,
        0.55,
        0.90,
        0.35,
        0.05,
        0.25,
        0.30,
        0.15,
    )
    best_scale = 1.0



In [3]:
submission = pd.DataFrame({"image_name": test_df["image_name"].values})

if train_df is None or "target" not in train_df.columns:
    submission["target"] = float(prior)
else:
    df_te = test_df.copy()

    def _norm_cat(s: pd.Series) -> pd.Series:
        s = s.astype("string")
        s = s.fillna("unknown")
        s = s.str.strip()
        s = s.mask(s == "", "unknown")
        return s

    for c in ["sex", "anatom_site_general_challenge", "patient_id"]:
        if c in df_te.columns:
            df_te[c] = _norm_cat(df_te[c])

    if "diagnosis" not in df_te.columns:
        df_te["diagnosis"] = "unknown"
    df_te["diagnosis"] = _norm_cat(df_te["diagnosis"])
    diag_missing_or_constant = bool(df_te["diagnosis"].nunique(dropna=False) <= 1)

    if "sex" in df_te.columns and "anatom_site_general_challenge" in df_te.columns:
        df_te["sex_site"] = df_te["sex"] + "||" + df_te["anatom_site_general_challenge"]
    else:
        df_te["sex_site"] = "unknown||unknown"

    if (
        "patient_id" in df_te.columns
        and "anatom_site_general_challenge" in df_te.columns
    ):
        df_te["pid_site"] = (
            df_te["patient_id"] + "||" + df_te["anatom_site_general_challenge"]
        )
    else:
        df_te["pid_site"] = "unknown||unknown"

    def logit(p):
        p = np.clip(p, 1e-6, 1 - 1e-6)
        return np.log(p / (1 - p))

    def sigmoid(x):
        return 1.0 / (1.0 + np.exp(-x))

    base = logit(prior)

    sex_p = (
        df_te["sex"].map(sex_map_full).astype(float).fillna(prior)
        if "sex" in df_te.columns
        else prior
    )
    site_p = (
        df_te["anatom_site_general_challenge"]
        .map(site_map_full)
        .astype(float)
        .fillna(prior)
        if "anatom_site_general_challenge" in df_te.columns
        else prior
    )
    sexsite_p = df_te["sex_site"].map(sexsite_map_full).astype(float).fillna(prior)

    if "patient_id" in df_te.columns and len(pid_map_full) > 0:
        pid_seen_rate = float(
            df_te["patient_id"].isin(pd.Index(list(pid_map_full.keys()))).mean()
        )
    else:
        pid_seen_rate = 0.0
    pid_disabled_test = bool(pid_seen_rate < 0.05)

    pid_p = (
        df_te["patient_id"].map(pid_map_full).astype(float).fillna(prior)
        if ("patient_id" in df_te.columns and not pid_disabled_test)
        else prior
    )

    diag_p = df_te["diagnosis"].map(diag_map_full).astype(float).fillna(prior)

    if "pid_site" in df_te.columns and len(pidsite_map_full) > 0:
        pidsite_seen_rate = float(
            df_te["pid_site"].isin(pd.Index(list(pidsite_map_full.keys()))).mean()
        )
    else:
        pidsite_seen_rate = 0.0
    pidsite_disabled_test = bool(pidsite_seen_rate < 0.05)

    pidsite_p = (
        df_te["pid_site"].map(pidsite_map_full).astype(float).fillna(prior)
        if ("pid_site" in df_te.columns and not pidsite_disabled_test)
        else prior
    )

    age_te = pd.to_numeric(
        df_te.get("age_approx", pd.Series([np.nan] * len(df_te))), errors="coerce"
    ).fillna(age_mean)
    age_z = (age_te - age_mean) / age_std

    w_diag_eff = 0.0 if diag_missing_or_constant else float(w_diag)
    w_pid_eff = 0.0 if pid_disabled_test else float(w_pid)
    w_pidsite_eff = 0.0 if pidsite_disabled_test else float(w_pidsite)

    if (
        age_bin_edges is not None
        and len(age_bin_edges) > 0
        and len(agebin_map_full) > 0
    ):
        right_edges = [iv.right for iv in age_bin_edges]
        bin_edges = [-np.inf] + right_edges
        age_bin_te = pd.cut(age_te, bins=bin_edges, labels=False, include_lowest=True)
        age_bin_te = age_bin_te.fillna(0).astype(int)
        agebin_p = (
            age_bin_te.astype("string").map(agebin_map_full).astype(float).fillna(prior)
        )

        if (
            "anatom_site_general_challenge" in df_te.columns
            and len(siteage_map_full) > 0
        ):
            site_agebin_te = (
                df_te["anatom_site_general_challenge"]
                + "||"
                + age_bin_te.astype("string")
            )
            siteage_p = site_agebin_te.map(siteage_map_full).astype(float).fillna(prior)
        else:
            siteage_p = prior

        score = (
            base
            + float(w_sex) * (logit(sex_p) - base)
            + float(w_site) * (logit(site_p) - base)
            + float(w_sexsite) * (logit(sexsite_p) - base)
            + float(w_agebin) * (logit(agebin_p) - base)
            + float(w_pid_eff) * (logit(pid_p) - base)
            + float(w_diag_eff) * (logit(diag_p) - base)
            + float(w_agez) * age_slope * age_z.values
            + float(w_siteage) * (logit(siteage_p) - base)
            + float(w_pidsite_eff) * (logit(pidsite_p) - base)
        )
    else:
        score = (
            base
            + float(w_sex) * (logit(sex_p) - base)
            + float(w_site) * (logit(site_p) - base)
            + float(w_sexsite) * (logit(sexsite_p) - base)
            + float(w_pid_eff) * (logit(pid_p) - base)
            + float(w_diag_eff) * (logit(diag_p) - base)
            + float(w_agez) * age_slope * age_z.values
            + float(w_pidsite_eff) * (logit(pidsite_p) - base)
        )

    pred = sigmoid(float(best_scale) * score)
    submission["target"] = np.clip(pred.astype(float), 1e-6, 1 - 1e-6)

if submission.shape[0] != test_df.shape[0]:
    raise ValueError("Submission row count does not match test row count.")

if not np.array_equal(submission["image_name"].values, test_df["image_name"].values):
    submission = (
        submission.set_index("image_name").reindex(test_df["image_name"]).reset_index()
    )



In [4]:
submission.to_csv("submission.csv", index=False, float_format="%.6f")

print("Wrote submission.csv")
print(submission.head())
print("prior used:", float(prior))
print("best_scale (OOF-fit):", float(best_scale))
print(
    "weights:",
    {
        "w_sex": float(w_sex),
        "w_site": float(w_site),
        "w_sexsite": float(w_sexsite),
        "w_agebin": float(w_agebin),
        "w_pid": float(w_pid),
        "w_diag": float(w_diag),
        "w_agez": float(w_agez),
        "w_siteage": float(w_siteage),
        "w_pidsite": float(w_pidsite),
        "w_pid_eff_test": float(
            0.0 if ("pid_disabled_test" in globals() and pid_disabled_test) else w_pid
        ),
        "w_pidsite_eff_test": float(
            0.0
            if ("pidsite_disabled_test" in globals() and pidsite_disabled_test)
            else w_pidsite
        ),
        "w_diag_eff_test": float(
            0.0
            if ("diag_missing_or_constant" in globals() and diag_missing_or_constant)
            else w_diag
        ),
    },
)
print(
    "pid_seen_rate_test:",
    float(pid_seen_rate) if "pid_seen_rate" in globals() else None,
    "pid_disabled_test:",
    bool(pid_disabled_test) if "pid_disabled_test" in globals() else None,
)
print(
    "pidsite_seen_rate_test:",
    float(pidsite_seen_rate) if "pidsite_seen_rate" in globals() else None,
    "pidsite_disabled_test:",
    bool(pidsite_disabled_test) if "pidsite_disabled_test" in globals() else None,
)
print(
    "target min/max:",
    float(submission["target"].min()),
    float(submission["target"].max()),
)

Wrote submission.csv
     image_name    target
0  ISIC_0052212  0.057168
1  ISIC_0076545  0.102397
2  ISIC_0085172  0.056905
3  ISIC_0086709  0.063349
4  ISIC_0109568  0.122808
prior used: 0.017699420369859234
best_scale (OOF-fit): 0.6
weights: {'w_sex': 0.45, 'w_site': 0.45, 'w_sexsite': 0.35, 'w_agebin': 0.65, 'w_pid': 0.5, 'w_diag': 0.1, 'w_agez': 0.15, 'w_siteage': 0.15, 'w_pidsite': 0.3, 'w_pid_eff_test': 0.5, 'w_pidsite_eff_test': 0.3, 'w_diag_eff_test': 0.0}
pid_seen_rate_test: 1.0 pid_disabled_test: False
pidsite_seen_rate_test: 0.9451955577015935 pidsite_disabled_test: False
target min/max: 0.04180245208639761 0.21681043648289092
